
### Notebook to compare the old and new photometry

We output a number of comparison plots, in addition to a manually curated final photometry file.



In [ ]:

""" reading in useful packages """

import numpy as np
import pandas as pd

import os

from datetime import date
from astropy.time import Time

import math

import matplotlib.pyplot as plt
import matplotlib.ticker as tck
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm

# setting figure style
plt.rcParams.update({
    "text.usetex" : False,
    "font.family" : "Arial", 
    "mathtext.fontset" : "stix",
    "font.size" : 12
})

fontsize = plt.rcParams["font.size"]


In [ ]:

""" defining transient name """

transient_name = "SN2025sei"




### Defining some useful functions



In [ ]:

def _mkdir(newdir):
    """ Creates the required directory for the output files. """
    """works the way a good mkdir should :)
        - already exists, silently complete
        - regular file in the way, raise an exception
        - parent directory(ies) does not exist, make them as well
    """
    if os.path.isdir(newdir):
        pass
    elif os.path.isfile(newdir):
        raise OSError("a file with the same name as the desired " \
                      "dir, '%s', already exists." % newdir)
    else:
        head, tail = os.path.split(newdir)
        if head and not os.path.isdir(head):
            _mkdir(head)
        #print "_mkdir %s" % repr(newdir)
        if tail:
            os.mkdir(newdir)
    return newdir


In [ ]:

def fnu2ABmag(flux):
    # simple function to convert flux --> AB mag
    mag = -2.5 * np.log10(flux) + 23.90 # flux in micro-Jy
    return mag


def ABmag2fnu(mag):
    # simple function to convert AB mag --> flux
    flux = 10**((mag - 23.90) / (-2.5))
    return flux


def efnu2eABmag(flux, eflux):
    # simple function to convert flux error --> AB mag error
    emag = (2.5 / np.log(10)) * (eflux / flux)
    return emag

def eABmag2efnu(mag, emag):
    # simple function to convert AB mag error --> flux error
    eflux = (
        (np.log(10) / 2.5)
        * 10**((mag - 23.90) / (-2.5))
        * emag
    )
    return eflux


In [ ]:

""" reading in Pan-STARRS photometry file """

today = date.today() # NOTE. Only works if run on the same day as the previous script

PS_phot = pd.read_csv(f"{_mkdir('FinalPhotometry/csv_files')}/TMP_{transient_name}_Pan-STARRS_photometry_{today}.csv", sep=",")

PS_phot


In [ ]:

""" converting MJDs --> standard date-time UTC format """

MJDs = Time(PS_phot["mjd"], format='mjd')

PS_phot["DateTime(UTC)"] = MJDs.iso

PS_phot



### [OPTIONAL] Manually curate the final photometry file; i.e., manually remove bad measurements related to e.g., satellite trails, artefacts etc.



In [ ]:

# """ clipping bad data points """

# # 60919.34566 | r --- impacted by a satellite trail

# indexes = PS_phot[(PS_phot["mjd"] == 60919.34566) & (PS_phot["filter"] == "r")].index.values[0]

# print(PS_phot.iloc[indexes])

# PS_phot.drop(indexes, inplace=True)
# PS_phot.reset_index(drop=True, inplace=True)

# PS_phot


In [ ]:

""" outputting a final column containing a mix of detections and upper limits """

mag_values_threesigma = []
emag_values_threesigma = []
upper_limit_boolean = []

for aa in range(len(PS_phot)):
    tmp_df = PS_phot.iloc[aa]

    if math.isnan(tmp_df["Mag_UL(3-sigma)"]) is False:
        mag_values_threesigma.append(tmp_df["Mag_UL(3-sigma)"])
        emag_values_threesigma.append(np.nan)
        upper_limit_boolean.append("True")
    else:
        mag_values_threesigma.append(tmp_df["Mag(AB)"])
        emag_values_threesigma.append(tmp_df["eMag(AB)"])
        upper_limit_boolean.append("False")



PS_phot["Final_Mag(AB)"] = mag_values_threesigma
PS_phot["Final_eMag(AB)"] = emag_values_threesigma
PS_phot["UL?"] = upper_limit_boolean

PS_phot["Final_rounded_Mag(AB)"] = PS_phot["Final_Mag(AB)"].round(2)
PS_phot["Final_rounded_eMag(AB)"] = PS_phot["Final_eMag(AB)"].round(2)

PS_phot


In [ ]:

""" saving final curated photometry file """

PS_phot.to_csv(f"{_mkdir('FinalPhotometry/csv_files')}/{transient_name}_Pan-STARRS_photometry_ZP-corrected_{today}.csv", sep=",")

PS_phot



### All of the below is for plotting purposes



In [ ]:

""" splitting the dataframe by filter - useful for plotting purposes """

PS_phot_g = PS_phot[PS_phot["filter"] == "g"].reset_index(drop=True)
PS_phot_r = PS_phot[PS_phot["filter"] == "r"].reset_index(drop=True)
PS_phot_i = PS_phot[PS_phot["filter"] == "i"].reset_index(drop=True)
PS_phot_z = PS_phot[PS_phot["filter"] == "z"].reset_index(drop=True)
PS_phot_y = PS_phot[PS_phot["filter"] == "y"].reset_index(drop=True)

PS_phot_i


In [ ]:

""" reading in the default Pan-STARRS forced photometry table -- for comparison """

forced_phot_df = pd.read_csv("Bundles/forced_phot_2026-05-05.txt", sep="\s+")

forced_phot_df = forced_phot_df.rename(columns={"#mjd" : "mjd"})

# rounding the MJD column entries to match the 3 d.p. in the offsets df loaded above
forced_phot_df["mjd"] = forced_phot_df["mjd"].round(3)

forced_phot_df["cal_psf_mag"] = forced_phot_df["cal_psf_mag"].astype(float)

# forced_phot_df.dtypes
# forced_phot_df


In [ ]:

""" splitting the dataframe by filter - useful for plotting purposes """

forced_phot_g = forced_phot_df[forced_phot_df["filter"] == "g"].reset_index(drop=True)
forced_phot_r = forced_phot_df[forced_phot_df["filter"] == "r"].reset_index(drop=True)
forced_phot_i = forced_phot_df[forced_phot_df["filter"] == "i"].reset_index(drop=True)
forced_phot_z = forced_phot_df[forced_phot_df["filter"] == "z"].reset_index(drop=True)
forced_phot_y = forced_phot_df[forced_phot_df["filter"] == "y"].reset_index(drop=True)

forced_phot_i



### Making some figures



In [ ]:

""" plotting a comparison between the old and new photometry -- all filters """

colorlist = ["blue", "green", "darkorange", "red", "brown"]
colorlist_counter = 0

markersize = 8
mjd_offset = 0.3

forced_phots_list = [
    forced_phot_g,
    forced_phot_r,
    forced_phot_i,
    forced_phot_z,
    forced_phot_y,
]

PS_phots_list = [
    PS_phot_g,
    PS_phot_r,
    PS_phot_i,
    PS_phot_z,
    PS_phot_y,
]


# Creating figure
fig = plt.figure(figsize=(24, 6))

# defining the rest wavelength axis
ax_mag = plt.subplot()

ax_mag.tick_params(axis='both', direction='in', top=True, right=False, left=True, bottom=True, which='both')
ax_mag.xaxis.set_minor_locator(tck.AutoMinorLocator())
ax_mag.yaxis.set_minor_locator(tck.AutoMinorLocator())
ax_mag.set_axisbelow(False)

# defining the observed wavelength axis
ax_fnu = ax_mag.twinx()

ax_fnu.tick_params(axis='both', direction='in', top=False, right=True, left=False, bottom=False, which='both')
ax_fnu.xaxis.set_minor_locator(tck.AutoMinorLocator())
ax_fnu.yaxis.set_minor_locator(tck.AutoMinorLocator())


for forced_phots, PS_phots in zip(forced_phots_list, PS_phots_list):

    for aa in range(len(PS_phots)):
        tmp_df = PS_phots.iloc[aa]

        mjd = tmp_df["mjd"]
        mag = tmp_df["Final_Mag(AB)"]
        emag = tmp_df["Final_eMag(AB)"]
        is_UL = tmp_df["UL?"]

        filter = tmp_df["filter"]

        if is_UL is "True":
            marker = "v"
        else:
            marker = "o"
        
        ax_mag.plot(mjd, mag, label=r"", color=colorlist[colorlist_counter], marker=marker, markersize=markersize, linestyle='', alpha=0.7, zorder=9, markeredgewidth=0.0)
        ax_mag.errorbar(mjd, mag, yerr=emag, ecolor=colorlist[colorlist_counter], linestyle='', alpha=0.7)

    ax_mag.plot(forced_phots["mjd"]-mjd_offset, forced_phots["cal_psf_mag"], label=r"", color=colorlist[colorlist_counter], marker="o", markersize=markersize, linestyle='', alpha=0.3, zorder=9, markeredgewidth=0.0)
    ax_mag.errorbar(forced_phots["mjd"]-mjd_offset, forced_phots["cal_psf_mag"], yerr=forced_phots["psf_inst_mag_sig"], ecolor=colorlist[colorlist_counter], linestyle='', alpha=0.3)

    colorlist_counter += 1


ax_mag.plot(60000, 25, color="black", marker="o", linestyle='', label="Detection")
ax_mag.plot(60000, 25, color="black", marker="v", linestyle='',  label="Upper limit ($3 \sigma$)")

ax_mag.plot(60000, 25, color=colorlist[0],  marker="o", linestyle='', label="$g$")
ax_mag.plot(60000, 25, color=colorlist[1],  marker="o", linestyle='', label="$r$")
ax_mag.plot(60000, 25, color=colorlist[2],  marker="o", linestyle='', label="$i$")
ax_mag.plot(60000, 25, color=colorlist[3],  marker="o", linestyle='', label="$z$")
ax_mag.plot(60000, 25, color=colorlist[4],  marker="o", linestyle='', label="$y$")

ax_mag.plot(60000, 25, color="black", marker="o", linestyle='', alpha=1, label="Zero-point corrected")
ax_mag.plot(60000, 25, color="black", marker="o", linestyle='', alpha=0.3, label="Default photometry")

ax_mag.set_xlim([60885, 60985])
ax_mag.set_ylim([23, 17.6])

ax_mag.grid(which='both', axis='x', linestyle="--", color="black", zorder=-999)

# First, we fix the axis range for the rest wavelength values
# This stops any last-minute re-scaling effects when other stuff is plotted
y_1, y_2 = ax_mag.get_ylim()
ax_fnu.set_ylim(y_1, y_2)

y_1 = ABmag2fnu(y_1)
y_2 = ABmag2fnu(y_2)

ax_fnu.set_ylim(y_1, y_2)
ax_fnu.set_yscale("log")

ax_mag.legend(frameon=True, edgecolor="white", loc="best", ncol=3, fontsize=fontsize, handletextpad=0.5)

ax_mag.set_xlabel("MJD + offset")
ax_mag.set_ylabel("Apparent magnitude (AB mag)")
ax_fnu.set_ylabel("Observed flux density ($\mu$Jy)")

ax_mag.set_title(f"Pan-STARRS photometry of {transient_name}")

plt.savefig(f"{_mkdir('FinalPhotometry/pdf_files')}/{transient_name}_Pan-STARRS_photometry_comparison-pre+post-ZP-corrected_ALL.pdf", dpi=900, bbox_inches="tight")
plt.show()


In [ ]:

""" plotting a comparison between the old and new photometry -- individual filters """

colorlist = ["blue", "green", "darkorange", "red", "brown"]
colorlist_counter = 0

markersize = 8
mjd_offset = 0.3

forced_phots_list = [
    forced_phot_g,
    forced_phot_r,
    forced_phot_i,
    forced_phot_z,
    forced_phot_y,
]

PS_phots_list = [
    PS_phot_g,
    PS_phot_r,
    PS_phot_i,
    PS_phot_z,
    PS_phot_y,
]


for forced_phots, PS_phots in zip(forced_phots_list, PS_phots_list):

    # Creating figure
    fig = plt.figure(figsize=(24, 6))

    # defining the rest wavelength axis
    ax_mag = plt.subplot()
    ax_mag.tick_params(axis='both', direction='in', top=True, right=False, left=True, bottom=True, which='both')
    ax_mag.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax_mag.yaxis.set_minor_locator(tck.AutoMinorLocator())

    ax_mag.set_axisbelow(False)

    # defining the observed wavelength axis
    ax_fnu = ax_mag.twinx()
    ax_fnu.tick_params(axis='both', direction='in', top=False, right=True, left=False, bottom=False, which='both')
    ax_fnu.xaxis.set_minor_locator(tck.AutoMinorLocator())
    ax_fnu.yaxis.set_minor_locator(tck.AutoMinorLocator())


    for aa in range(len(PS_phots)):
        tmp_df = PS_phots.iloc[aa]

        mjd = tmp_df["mjd"]
        mag = tmp_df["Final_Mag(AB)"]
        emag = tmp_df["Final_eMag(AB)"]
        is_UL = tmp_df["UL?"]

        filter = tmp_df["filter"]

        if is_UL is "True":
            marker = "v"
        else:
            marker = "o"
        
        ax_mag.plot(mjd, mag, label=r"", color=colorlist[colorlist_counter], marker=marker, markersize=markersize, linestyle='', alpha=0.7, zorder=9, markeredgewidth=0.0)
        ax_mag.errorbar(mjd, mag, yerr=emag, ecolor=colorlist[colorlist_counter], linestyle='', alpha=0.7)

    ax_mag.plot(forced_phots["mjd"]-mjd_offset, forced_phots["cal_psf_mag"], label=r"", color=colorlist[colorlist_counter], marker="o", markersize=markersize, linestyle='', alpha=0.3, zorder=9, markeredgewidth=0.0)
    ax_mag.errorbar(forced_phots["mjd"]-mjd_offset, forced_phots["cal_psf_mag"], yerr=forced_phots["psf_inst_mag_sig"], ecolor=colorlist[colorlist_counter], linestyle='', alpha=0.3)

    #######################################################################################################################
    #######################################################################################################################


    ax_mag.plot(60000, 25, color="black", marker="o", linestyle='', label="Detection")
    ax_mag.plot(60000, 25, color="black", marker="v", linestyle='',  label="Upper limit ($3 \sigma$)")

    ax_mag.plot(60000, 25, color=colorlist[colorlist_counter],  marker="o", linestyle='', label=rf"${filter}$")

    colorlist_counter += 1

    ax_mag.plot(60000, 25, color="black", marker="o", linestyle='', alpha=1, label="Zero-point corrected")
    ax_mag.plot(60000, 25, color="black", marker="o", linestyle='', alpha=0.3, label="Default photometry")

    ax_mag.set_xlim([60885, 60985])
    ax_mag.set_ylim([23, 17.6])

    ax_mag.grid(which='both', axis='x', linestyle="--", color="black", zorder=-999)

    # First, we fix the axis range for the rest wavelength values
    # This stops any last-minute re-scaling effects when other stuff is plotted
    y_1, y_2 = ax_mag.get_ylim()
    ax_fnu.set_ylim(y_1, y_2)

    y_1 = ABmag2fnu(y_1)
    y_2 = ABmag2fnu(y_2)
    ax_fnu.set_ylim(y_1, y_2)

    ax_fnu.set_yscale("log")

    ax_mag.legend(frameon=True, edgecolor="white", loc="best", ncol=2, fontsize=fontsize, handletextpad=0.5)

    ax_mag.set_xlabel("MJD + offset")
    ax_mag.set_ylabel("Apparent magnitude (AB mag)")
    ax_fnu.set_ylabel("Observed flux density ($\mu$Jy)")

    ax_mag.set_title(f"Pan-STARRS photometry of {transient_name}")

    plt.savefig(f"{_mkdir('FinalPhotometry/pdf_files')}/{transient_name}_Pan-STARRS_photometry_comparison-pre+post-ZP-corrected_{filter}.pdf", dpi=900, bbox_inches="tight")
    plt.show()
